# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login

login()

In [2]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, duckdb, pandas as pd, numpy as np
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"
os.makedirs(BASE, exist_ok=True)

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

Mounted at /content/drive


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, os, json ,duckdb
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"

data_model = pd.read_csv(f"{BASE}/data_model_v2.csv")
print("Loaded:", data_model.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (183345, 19)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

model_cols = ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
              "february_clicks", "click_through_rate", "weighted_position", "momentum",
              "active_days", "click_through_rate_missing", "weighted_position_missing", "momentum_missing"]

X = data_model[model_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

# Use the SAME grouped split as Week 6 for consistency
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

# Score EVERY page (not just test set) — the playbook needs to cover the whole portfolio
data_model["risk_score"] = rf.predict_proba(X)[:, 1]
print("Scored all pages.")

Scored all pages.


# Note:

  the playbook needs a score for every page a content team might review, so we score the full dataset here, using the model trained only on the training half — the test half was for measuring honesty, this step is for producing the actual deliverable.

In [6]:
# Reason codes — built from the same confirmed signals across Weeks 4-6
def reason_code(row):
    momentum_bad = row["momentum"] < -0.1          # declining trend April vs Feb
    ctr_bad = row["click_through_rate"] < data_model["click_through_rate"].median()
    no_history = row["weighted_position_missing"] == 1

    if no_history:
        return "INSUFFICIENT_HISTORY"
    elif momentum_bad and ctr_bad:
        return "DECLINING_AND_LOW_CTR"
    elif momentum_bad:
        return "DECLINING_TREND"
    elif ctr_bad:
        return "LOW_CTR_ONLY"
    else:
        return "MONITOR"

data_model["reason_code"] = data_model.apply(reason_code, axis=1)

action_map = {
    "DECLINING_AND_LOW_CTR": "refresh_now",
    "DECLINING_TREND": "refresh",
    "LOW_CTR_ONLY": "fix_snippet_or_title",
    "INSUFFICIENT_HISTORY": "monitor_gather_data",
    "MONITOR": "monitor"
}
data_model["action"] = data_model["reason_code"].map(action_map)

ranked_queue = data_model.sort_values("risk_score", ascending=False).reset_index(drop=True)
ranked_queue[["client_hash_id", "content_hash_id", "risk_score", "reason_code", "action"]].head(20)

,client_hash_id,content_hash_id,risk_score,reason_code,action
0,client_08a6a72ff48e62c0,content_c52c9aad099ec5cb,0.955337,MONITOR,monitor
1,client_08a6a72ff48e62c0,content_470153fea60157cf,0.955337,MONITOR,monitor
2,client_08a6a72ff48e62c0,content_522b0aa738196dea,0.955337,MONITOR,monitor
3,client_08a6a72ff48e62c0,content_8109e1f8ad5b2d82,0.955337,MONITOR,monitor
4,client_08a6a72ff48e62c0,content_4218b21da2a821fa,0.955337,MONITOR,monitor
5,client_08a6a72ff48e62c0,content_652f609554323ee9,0.955337,MONITOR,monitor
6,client_08a6a72ff48e62c0,content_7b8a875235c117d1,0.955337,MONITOR,monitor
7,client_08a6a72ff48e62c0,content_16572346ba5fef5f,0.955337,MONITOR,monitor
8,client_08a6a72ff48e62c0,content_73e5d22d34811e01,0.955337,MONITOR,monitor
9,client_08a6a72ff48e62c0,content_0bdc6436ecb19165,0.955337,MONITOR,monitor


In [8]:
def reason_code_v2(row):
    if row["click_through_rate"] == 0 and row["impressions_window"] < 10:  # tune threshold as needed
        return "ZERO_TRAFFIC_DEAD_PAGE"
    # ... rest of existing logic

# Note:

 reason_code translates model math into a human-readable "why" — nobody wants to act on a bare number, they want to know what's actually going on with the page.



The queue, in words a human trusts: each page gets a risk score (0-1), a reason code explaining what's driving that score, and a suggested action. A page marked DECLINING_AND_LOW_CTR with refresh_now means: this page's clicks are trending down April-vs-February AND it's underperforming its expected click rate — the two strongest signals confirmed across Weeks 4-6 firing together. A page marked INSUFFICIENT_HISTORY means the model genuinely doesn't have enough data to judge it confidently — that's flagged honestly as its own category, not silently guessed at.

In [9]:
def reason_code_v2(row):
    if row["click_through_rate"] == 0 and row["impressions_window"] < 10:
        return "ZERO_TRAFFIC_DEAD_PAGE"
    momentum_bad = row["momentum"] < -0.1
    ctr_bad = row["click_through_rate"] < data_model["click_through_rate"].median()
    no_history = row["weighted_position_missing"] == 1

    if no_history:
        return "INSUFFICIENT_HISTORY"
    elif momentum_bad and ctr_bad:
        return "DECLINING_AND_LOW_CTR"
    elif momentum_bad:
        return "DECLINING_TREND"
    elif ctr_bad:
        return "LOW_CTR_ONLY"
    else:
        return "MONITOR"

data_model["reason_code"] = data_model.apply(reason_code_v2, axis=1)
data_model["action"] = data_model["reason_code"].map({
    **action_map,
    "ZERO_TRAFFIC_DEAD_PAGE": "review_or_deprioritize"
})

print(data_model["reason_code"].value_counts())

reason_code
INSUFFICIENT_HISTORY      73969
DECLINING_TREND           70593
MONITOR                   21080
ZERO_TRAFFIC_DEAD_PAGE    14665
DECLINING_AND_LOW_CTR      2678
LOW_CTR_ONLY                360
Name: count, dtype: int64


In [10]:
print("Zero-traffic dead pages:", (data_model["reason_code"] == "ZERO_TRAFFIC_DEAD_PAGE").sum())

Zero-traffic dead pages: 14665


In [11]:
print(data_model["reason_code"].value_counts())
print()
print(data_model["reason_code"].value_counts(normalize=True))

reason_code
INSUFFICIENT_HISTORY      73969
DECLINING_TREND           70593
MONITOR                   21080
ZERO_TRAFFIC_DEAD_PAGE    14665
DECLINING_AND_LOW_CTR      2678
LOW_CTR_ONLY                360
Name: count, dtype: int64

reason_code
INSUFFICIENT_HISTORY      0.403442
DECLINING_TREND           0.385028
MONITOR                   0.114975
ZERO_TRAFFIC_DEAD_PAGE    0.079986
DECLINING_AND_LOW_CTR     0.014606
LOW_CTR_ONLY              0.001964
Name: proportion, dtype: float64


In [12]:
print(data_model["momentum"].describe())
print((data_model["momentum"] < -0.1).mean())

count    183345.000000
mean         -0.276047
std           1.689756
min          -1.000000
25%          -0.481481
50%          -0.481481
75%          -0.481481
max         263.000000
Name: momentum, dtype: float64
0.8830619869644659


In [13]:
print(data_model[data_model["reason_code"]=="ZERO_TRAFFIC_DEAD_PAGE"]["client_hash_id"].nunique(), "distinct clients affected")

47 distinct clients affected


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Expected columns should roughly be:

2 ID columns (client_hash_id, content_hash_id)
~9 raw features (impressions_window, clicks_window, april_impressions, april_clicks, february_clicks, click_through_rate, weighted_position, momentum, active_days)
3 missing-flags (click_through_rate_missing, weighted_position_missing, momentum_missing)
2-3 label-related columns (clicks_april, clicks_may, declined)
possibly baseline_score if that got merged in too

That adds up to somewhere around 18-20, so 19 is a very plausible, healthy number — not a sign of anything missing or duplicated.

The real confidence check, though, is the one from my last message — not just the shape, but whether retraining reproduces Week 6's exact AUC/precision numbers. Shape matching is a good first sanity check, but it doesn't guarantee the model behaves identically — run the assert check to get that actual proof. Want to run that now?

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.